In [0]:
SELECT
    municipality_code,
    city,
    COUNT(*) AS posting_count
FROM silver_jobtech.job_postings_silver_clean
WHERE municipality_code IS NOT NULL
GROUP BY municipality_code, city
ORDER BY posting_count DESC
LIMIT 50;

## Municipality to län mapping

This table maps Swedish municipality codes to their corresponding län (county).

### Purpose
This mapping enables integration between:
- JobTech data (municipality level)
- SCB data (län level)

### Source
Based on official Swedish municipality and county codes.

### Usage
This table will be used to build `dim_location` in the Gold layer.

In [0]:
CREATE OR REPLACE TABLE gold_analysis.municipality_lan_lookup AS
SELECT * FROM VALUES
('0180', 'Stockholm', '01', 'Stockholms län'),
('0184', 'Solna', '01', 'Stockholms län'),
('0181', 'Södertälje', '01', 'Stockholms län'),
('0182', 'Nacka', '01', 'Stockholms län'),

('1480', 'Göteborg', '14', 'Västra Götalands län'),
('1481', 'Mölndal', '14', 'Västra Götalands län'),
('1490', 'Borås', '14', 'Västra Götalands län'),

('1280', 'Malmö', '12', 'Skåne län'),
('1281', 'Lund', '12', 'Skåne län'),
('1283', 'Helsingborg', '12', 'Skåne län'),

('0380', 'Uppsala', '03', 'Uppsala län'),
('0580', 'Linköping', '05', 'Östergötlands län'),
('0581', 'Norrköping', '05', 'Östergötlands län'),

('0680', 'Jönköping', '06', 'Jönköpings län'),

('0780', 'Växjö', '07', 'Kronobergs län'),

('1880', 'Örebro', '18', 'Örebro län'),

('1980', 'Västerås', '19', 'Västmanlands län'),

('2180', 'Gävle', '21', 'Gävleborgs län'),

('2480', 'Umeå', '24', 'Västerbottens län')

AS t(municipality_code, municipality_name, lan_code, lan_name);

In [0]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN l.lan_code IS NOT NULL THEN 1 ELSE 0 END) AS mapped_rows,
    ROUND(
        100.0 * SUM(CASE WHEN l.lan_code IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS coverage_pct
FROM silver_jobtech.job_postings_silver_clean j
LEFT JOIN gold_analysis.municipality_lan_lookup l
    ON j.municipality_code = l.municipality_code;

The coverage test above means:

- ~3.6M rows mapped
- ~3.1M rows NOT mapped

Because we only used a small subset of municipalities.

To build full municipality to region mapping our look up should have all 290 municipalities for a ~100% coverage.


In [0]:
CREATE OR REPLACE TABLE gold_analysis.municipality_lan_lookup_full AS
SELECT DISTINCT
    municipality_code,
    SUBSTRING(municipality_code, 1, 2) AS lan_code
FROM silver_jobtech.job_postings_silver_clean
WHERE municipality_code IS NOT NULL;

In [0]:
CREATE OR REPLACE TABLE gold_analysis.lan_lookup AS
SELECT * FROM VALUES
('01', 'Stockholms län'),
('03', 'Uppsala län'),
('04', 'Södermanlands län'),
('05', 'Östergötlands län'),
('06', 'Jönköpings län'),
('07', 'Kronobergs län'),
('08', 'Kalmar län'),
('09', 'Gotlands län'),
('10', 'Blekinge län'),
('12', 'Skåne län'),
('13', 'Hallands län'),
('14', 'Västra Götalands län'),
('17', 'Värmlands län'),
('18', 'Örebro län'),
('19', 'Västmanlands län'),
('20', 'Dalarnas län'),
('21', 'Gävleborgs län'),
('22', 'Västernorrlands län'),
('23', 'Jämtlands län'),
('24', 'Västerbottens län'),
('25', 'Norrbottens län')
AS t(lan_code, lan_name);

In [0]:
CREATE OR REPLACE TABLE gold_analysis.municipality_lan_lookup AS
SELECT
    m.municipality_code,
    m.lan_code,
    l.lan_name
FROM gold_analysis.municipality_lan_lookup_full m
LEFT JOIN gold_analysis.lan_lookup l
    ON m.lan_code = l.lan_code;

In [0]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN l.lan_code IS NOT NULL THEN 1 ELSE 0 END) AS mapped_rows,
    ROUND(
        100.0 * SUM(CASE WHEN l.lan_code IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS coverage_pct
FROM silver_jobtech.job_postings_silver_clean j
LEFT JOIN gold_analysis.municipality_lan_lookup l
    ON j.municipality_code = l.municipality_code;

## Municipality to län mapping (derived)

Instead of manually mapping municipalities to counties (län),
we derive län_code directly from municipality_code.

### Logic
- Swedish municipality codes: first 2 digits = län_code
- Example:
  - 0180 → 01 → Stockholms län
  - 1480 → 14 → Västra Götalands län

### Result
- Coverage: 97.91%
- Successfully mapped the vast majority of job postings

### Remaining gaps
Unmapped rows are primarily due to:
- Missing municipality_code
- Rare edge cases in source data

### Conclusion
This mapping is sufficient for:
- Gold layer modeling
- Power BI analysis
- Cross-dataset integration with SCB data

In [0]:
DROP TABLE gold_analysis.municipality_lan_lookup_full;

### Note on lookup tables

Two tables were created during the mapping process:

- `municipality_lan_lookup_full` → intermediate (technical)
- `municipality_lan_lookup` → final (enriched)

Only the final table is used in downstream Gold models.